# Mushroom Classification with ViT


## 1. Налаштування середовища

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU не знайдено! Перейдіть в Runtime → Change runtime type → GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive підключено")

In [ ]:

import os
os.chdir('/content')

if os.path.exists('Applied_Mashroomatics'):
    os.chdir('Applied_Mashroomatics')
    !git pull
else:
    !git clone https://github.com/konstanta-asya/Applied_Mashroomatics.git
    os.chdir('Applied_Mashroomatics')

print(f"Робоча директорія: {os.getcwd()}")

In [ ]:
!pip install -q tqdm timm

In [ ]:
import os

!rm -rf data/raw
!mkdir -p data/raw

DATA_PATH = "/content/drive/MyDrive/mushroom_data"  # або "/content/drive/MyDrive/raw"

!ln -s {DATA_PATH}/DF20M data/raw/DF20M
!ln -s {DATA_PATH}/DF20M-metadata data/raw/DF20M-metadata

if os.path.exists('data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv'):
    print("Дані підключено успішно")
    !ls data/raw/DF20M-metadata/
else:
    print("Дані не знайдено! Перевірте шлях до папки на Google Drive")
    print(f"   Очікується: {DATA_PATH}/DF20M та {DATA_PATH}/DF20M-metadata")

## 2. Тренування моделі

**ViT + Metadata Fusion:**
- Модель: `vit_base_patch16_224` з metadata токеном
- Metadata: habitat, substrate, month (sinusoidal encoding)
- Backbone заморожений перші 5 епох, потім розморожується
- Differential LR: backbone=3e-5, meta/head=1e-4

**Оптимізації:**
- Mixed precision (AMP)
- Label smoothing (0.1)
- Gradient clipping (max_norm=1.0)
- Early stopping (patience=5)
- **RESUME** - продовжує з останнього checkpoint

In [ ]:
# === METADATA FUSION TRAINING (A100 optimized) ===
BATCH_SIZE = 128          # Більший batch для A100
EPOCHS = 30
NUM_WORKERS = 4           # Паралельне завантаження даних
RESUME = True

compile_flag = "--compile"
resume_flag = f"--resume /content/drive/MyDrive/mushroom_checkpoints/latest_checkpoint.pth" if RESUME else ""

!python src/train_vit.py \
    --csv_path data/raw/DF20M-metadata/DF20M-train_metadata_PROD.csv \
    --data_dir data/raw/DF20M \
    --batch_size {BATCH_SIZE} \
    --epochs {EPOCHS} \
    --amp \
    {compile_flag} \
    --num_workers {NUM_WORKERS} \
    --checkpoint_dir /content/drive/MyDrive/mushroom_checkpoints \
    {resume_flag}

## 3. Результати

In [ ]:
import os

checkpoint_dir = '/content/drive/MyDrive/mushroom_checkpoints'

if os.path.exists(checkpoint_dir):
    print("📁 Збережені файли:")
    for f in os.listdir(checkpoint_dir):
        size = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1e6
        print(f"  - {f} ({size:.1f} MB)")
else:
    print("❌ Папка з чекпоінтами не знайдена")

In [ ]:

from google.colab import files
import os

best_model = '/content/drive/MyDrive/mushroom_checkpoints/best_model.pth'

if os.path.exists(best_model):
    files.download(best_model)
    print("✅ Модель завантажено")
else:
    print("❌ Файл best_model.pth не знайдено. Тренування ще не завершено?")

## 4. Тестування моделі

In [ ]:
import torch
import sys
sys.path.insert(0, '/content/Applied_Mashroomatics')

from src.models.mushroom_vit import MushroomViTWithMetadata

checkpoint = torch.load('/content/drive/MyDrive/mushroom_checkpoints/best_model.pth')
num_classes = checkpoint['num_classes']
habitat_vocab = checkpoint['habitat_vocab']
substrate_vocab = checkpoint['substrate_vocab']

print(f"Model type: {checkpoint.get('model_type', 'unknown')}")
print(f"Кількість класів: {num_classes}")
print(f"Точність на валідації: {checkpoint['val_acc']:.2f}%")
print(f"Habitat categories: {len(habitat_vocab)}")
print(f"Substrate categories: {len(substrate_vocab)}")

model = MushroomViTWithMetadata(
    num_classes=num_classes,
    habitat_vocab=habitat_vocab,
    substrate_vocab=substrate_vocab,
    pretrained=False
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("Модель завантажено і готова до використання")

In [ ]:
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict(image_path, habitat=None, substrate=None, month=6):
    """
    Predict mushroom species with metadata.
    
    Args:
        image_path: Path to image
        habitat: Habitat string (e.g., 'forest') or None for UNK
        substrate: Substrate string (e.g., 'wood') or None for UNK  
        month: Month 1-12 (default 6 = June)
    """
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0)
    
    # Encode metadata
    habitat_id = torch.tensor([habitat_vocab.get(habitat, len(habitat_vocab))])
    substrate_id = torch.tensor([substrate_vocab.get(substrate, len(substrate_vocab))])
    month_tensor = torch.tensor([month])
    
    with torch.no_grad():
        output = model(input_tensor, habitat_id, substrate_id, month_tensor)
        probs = torch.softmax(output, dim=1)[0]
        top5_probs, top5_idx = torch.topk(probs, 5)
    
    print(f"Top-5 predictions:")
    for i, (prob, idx) in enumerate(zip(top5_probs, top5_idx)):
        print(f"  {i+1}. Class {idx.item()}: {prob.item()*100:.2f}%")
    
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.show()
    
    return top5_idx[0].item(), top5_probs[0].item()

# Example:
# predict('/content/drive/MyDrive/mushroom.jpg', habitat='forest', substrate='wood', month=9)